# Optimality gap analysis

## Load data


In [ ]:
from collections.abc import Sequence
from pathlib import Path

import matplotlib as mpl
import matplotlib.patches as patches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib.cm import ScalarMappable
from matplotlib.colors import BoundaryNorm, ListedColormap
from matplotlib.lines import Line2D
from matplotlib.ticker import FixedLocator
from PyComplexHeatmap import ClusterMapPlotter, HeatmapAnnotation, anno_barplot
from scipy.stats import mannwhitneyu, spearmanr

from lib import (
    N_OPTIMIZERS,
    N_PROBLEMS,
    N_RUNS,
    OPTIMIZER_OVERVIEW_PATH,
    PROBLEM_OVERVIEW_PATH,
    RELATIVE_WALLTIME_LIMIT,
    get_threshold,
)

optimizer_df = pd.read_csv(OPTIMIZER_OVERVIEW_PATH)
problem_df = pd.read_csv(PROBLEM_OVERVIEW_PATH)

df_marvin = pd.read_csv("../data/best_fx_marvin.csv")
df_ft3 = pd.read_csv("../data/best_fx_ft3.csv")

df = pd.concat([df_marvin, df_ft3], ignore_index=True)
df = df.join(
    problem_df.set_index(["short"]),
    on="problem",
    how="left",
    validate="many_to_one",
)
df = df.join(
    optimizer_df.set_index(["output_dir"]),
    on="optimizer",
    how="left",
    validate="many_to_one",
)
df = df.query("~excluded")
df.query("~is_pysacess")[
    ["problem", "optimizer", "optimizer_label", "run_idx", "fx_best", "site"]
].to_csv("out/endoints_per_run.csv", index=False)
assert len(df) > N_OPTIMIZERS * N_PROBLEMS * N_RUNS
assert np.isfinite(df.fx_best).all()
assert not df[["problem", "optimizer", "run_idx", "site"]].duplicated().any()

df

## Compute optimality gaps

In [ ]:
# site-specific and global optimality gaps
global_best_df_with_pyscat = df.groupby("problem")["fx_best"].min()
global_best_with_pyscat = global_best_df_with_pyscat.to_dict()

global_best_df_without_pyscat = (
    df.query("~is_pysacess").groupby("problem")["fx_best"].min()
)
global_best_without_pyscat = global_best_df_without_pyscat.to_dict()
site_best_df_without_pyscat = (
    df.query("~is_pysacess").groupby(["problem", "site"])["fx_best"].min()
)
site_best_df_with_pyscat = df.groupby(["problem", "site"])["fx_best"].min()

df["ref_global_without_pyscat"] = df["problem"].map(
    global_best_df_without_pyscat
)
df["ref_site_without_pyscat"] = df.set_index(["problem", "site"]).index.map(
    site_best_df_without_pyscat
)
df["ref_site_with_pyscat"] = df.set_index(["problem", "site"]).index.map(
    site_best_df_with_pyscat
)

df["optimality_gap_global_without_pyscat"] = (
    df["fx_best"] - df["ref_global_without_pyscat"]
)
df["optimality_gap_site_without_pyscat"] = (
    df["fx_best"] - df["ref_site_without_pyscat"]
)
df["optimality_gap_site_with_pyscat"] = (
    df["fx_best"] - df["ref_site_with_pyscat"]
)
df["optimality_gap_global_with_pyscat"] = df.apply(
    lambda row: row["fx_best"] - global_best_with_pyscat[row["problem"]],
    axis=1,
)

assert np.isfinite(df["optimality_gap_global_with_pyscat"]).all()
assert (df["optimality_gap_global_with_pyscat"] >= 0).all()
assert (df["optimality_gap_site_with_pyscat"] >= 0).all()
assert np.isfinite(df["optimality_gap_global_without_pyscat"]).all()
assert np.isfinite(df["optimality_gap_site_without_pyscat"]).all()
# we can have og<0 for pypscat results if we use the global best
#  from only the other optimizers
assert (
    df.query("~is_pysacess")["optimality_gap_global_without_pyscat"] >= 0
).all(), df.query("optimality_gap_global_without_pyscat < 0")

# offset to apply before log transform
#  use something small enough so we can still distinguish below or above threshold
log_offset = 1e-8
df["optimality_gap_global_with_pyscat_log10_offset"] = np.log10(
    df["optimality_gap_global_with_pyscat"] + log_offset
)

with np.errstate(invalid="ignore"):
    # see og<0 comment above
    df["optimality_gap_global_without_pyscat_log10_offset"] = np.log10(
        df["optimality_gap_global_without_pyscat"] + log_offset
    )
    df["optimality_gap_site_with_pyscat_log10_offset"] = np.log10(
        df["optimality_gap_site_with_pyscat"] + log_offset
    )
    df["optimality_gap_site_without_pyscat_log10_offset"] = np.log10(
        df["optimality_gap_site_without_pyscat"] + log_offset
    )

df.to_csv("out/optimality_gaps.csv", index=False)

df_best_fvals = pd.DataFrame(
    {
        # global
        "without_pyscat": global_best_df_without_pyscat,
        "with_pyscat": global_best_df_with_pyscat,
        # site-specific
        "marvin_without_pyscat": df.query("~is_pysacess and site == 'marvin'")
        .groupby("problem")["fx_best"]
        .min(),
    }
)
df_best_fvals.to_csv("out/best_fvals.csv")
df_best_fvals

In [ ]:
assert np.isfinite(df.fx_best).all(), df[~np.isfinite(df.fx_best)]

## Reliability plot



In [ ]:
site = "marvin"
df_tmp = df.query(f"~is_pysacess and site == '{site}'")
alpha = 0.05

# best/worst/... out of 10 runs
stats_df = (
    df_tmp.groupby(["optimizer", "problem"])[
        "optimality_gap_site_without_pyscat"
    ]
    .agg(best="min", worst="max", median="median", mean="mean")
    .lt(get_threshold(1 - alpha))
    .groupby("optimizer")
    .sum()
    .rename(
        columns={
            "best": "solved_best",
            "worst": "solved_worst",
            "median": "solved_median",
            "mean": "solved_mean",
        }
    )
    .join(optimizer_df.set_index("output_dir"))
    .sort_values(
        ["solved_best", "solved_median", "solved_worst"], ascending=False
    )
)
assert (stats_df["solved_best"] >= stats_df["solved_median"]).all()
assert (stats_df["solved_best"] >= stats_df["solved_mean"]).all()
stats_df

In [ ]:
fig, ax = plt.subplots(figsize=(18 / 2.54, 7), layout="constrained")

sns.scatterplot(
    stats_df,
    x="solved_best",
    y="optimizer_label",
    marker="D",
    facecolors="none",
    edgecolors="seagreen",
    label="Best of 10 runs",
    zorder=2,
    ax=ax,
)
sns.scatterplot(
    stats_df,
    x="solved_median",
    y="optimizer_label",
    marker="o",
    facecolors="none",
    s=plt.rcParams["lines.markersize"] * 20,
    edgecolors="steelblue",
    label="Median of 10 runs",
    zorder=2,
    ax=ax,
)
sns.scatterplot(
    stats_df,
    x="solved_worst",
    y="optimizer_label",
    marker="x",
    color="tomato",
    label="Worst of 10 runs",
    zorder=2,
    ax=ax,
)

for _, row in stats_df.iterrows():
    ax.plot(
        [row["solved_worst"], row["solved_best"]],
        [row["optimizer_label"], row["optimizer_label"]],
        color="gray",
        linewidth=1,
        zorder=0,
    )

ax.legend()
ax.set_xlim(0)
ax.set_xlabel(
    rf"Number of problems solved (OG ≤ {get_threshold(1 - alpha):.2f}, α = {alpha})"
)
ax.set_ylabel("Optimisation method")

ax.grid(which="major", axis="x", color="lightgrey", linewidth=0.5)

plt.savefig("out/reliability_plot.pdf")

## Performance profiles (solved problems over optimality gap)

In [ ]:
def plot_solved_problems_over_suboptimality(df, only_top_n=8):
    df = (
        df.query("site == 'marvin' and ~is_pysacess")
        .groupby(["problem", "optimizer"], as_index=False)
        .agg(
            optimality_gap=("optimality_gap_site_without_pyscat", "min"),
        )
        .join(
            optimizer_df.set_index(["output_dir"]),
            on="optimizer",
            how="left",
            validate="many_to_one",
        )
    )
    # rank by the number of problems solved at alpha=0.05
    n_solved_at_ref_df = (
        (
            df.set_index("optimizer_label").optimality_gap
            < get_threshold(1 - 0.05)
        )
        .groupby("optimizer_label")
        .sum()
        .sort_values(ascending=False)
    )
    if only_top_n:
        highlight_optimisers = n_solved_at_ref_df.nlargest(
            only_top_n
        ).index.values.tolist()

    x_max = max(1e4, df.optimality_gap.max())
    df["optimizer_label"] = pd.Categorical(
        df["optimizer_label"],
        categories=n_solved_at_ref_df.index.values,
        ordered=True,
    )

    fig, ax = plt.subplots()

    for i, (optimizer, grouped) in enumerate(
        df.groupby("optimizer_label", observed=False)
    ):
        grouped = grouped.sort_values("optimality_gap")
        x = grouped.optimality_gap.values
        if x[0] != 0:
            x = np.insert(x, 0, 0)  # add 0 at the beginning
        if x[-1] < x_max:
            x = np.append(x, x_max)
        # offset to avoid overplotting
        if optimizer in highlight_optimisers:
            offset = -(highlight_optimisers.index(optimizer) % 2 * 0.1)
        else:
            offset = 0
        # number of solved problems; steps at each optimality gap value
        y = (
            np.array([(grouped.optimality_gap <= og).sum() for og in x])
            + offset
        )
        if not only_top_n or optimizer in highlight_optimisers:
            ax.step(x, y, where="post", label=optimizer, color=f"C{i}")
        else:
            # low-ranking ones in the background
            ax.step(
                x,
                y,
                where="post",
                color="lightgrey",
                zorder=-1,
                linewidth=0.5 * plt.rcParams["lines.linewidth"],
            )
    ax.set_xscale("symlog", linthresh=1e-2)

    # generate minor tick positions across your x range
    minor_ticks = [0.1]
    for exp in range(-1, int(np.log10(x_max))):
        for sub in np.arange(2, 10):
            minor_ticks.append(sub * 10**exp)
    ax.set_xticks(minor_ticks, minor=True)
    ax.tick_params(axis="x", which="minor", length=2, color="black", width=0.8)

    ax.set_ylim(0, N_PROBLEMS)
    ax.set_xlabel("Optimality gap threshold")
    ax.set_ylabel("Number of problems solved")
    ax.legend(
        title=f"Top {only_top_n} optimisation methods at $\\alpha=0.05$\n(grey = others)"
        if only_top_n
        else None,
        bbox_to_anchor=(1.05, 1),
        loc=2,
        borderaxespad=0.0,
        ncols=1 if only_top_n else 2,
    )

    # plot thresholds
    for alpha in [0.05, 0.01, 0.001]:
        thresh = get_threshold(percentile=1 - alpha)
        ax.axvline(
            thresh,
            color="gray",
            linestyle="--",
            label=f"chi2 threshold (alpha={alpha})",
        )
        # label next to each line
        ax.text(
            thresh,
            ax.get_ylim()[1] * 0.02,
            f"$\\alpha={alpha}$, $OG={thresh:.2f}$",
            fontsize="small",
            rotation=90,
            verticalalignment="bottom",
            horizontalalignment="right",
        )

    ax.set_xlim(right=1e3)


with plt.rc_context(
    {
        "figure.dpi": 300,
        "figure.figsize": (6.5, 4),
    }
):
    plot_solved_problems_over_suboptimality(df)
plt.savefig("out/performance_profiles_marvin.pdf", bbox_inches="tight")

## Ranking by best_fx

No cutoff, just ranking by final value of each run. Average of runs.

Fuzzy ranking. Average rank in case of ties.

In [ ]:
def fuzzy_rank(series, tolerance, ascending=True, method="min"):
    """
    Assign ranks to a series, treating values within `tolerance` as equal.
    method: "min" (competition ranking: 1,1,3) or "average" (1.5,1.5,3)
    """
    sorted_vals = series.sort_values(ascending=ascending)
    gaps = sorted_vals.diff().abs().fillna(0)
    group_id = (gaps > tolerance).cumsum()

    group_sizes = group_id.value_counts().sort_index()
    group_start_rank = group_sizes.cumsum().shift(1).fillna(0).astype(int) + 1

    if method == "min":
        rank_map = group_id.map(group_start_rank)
    elif method == "average":
        # average rank = start_rank + (size - 1) / 2
        group_avg_rank = group_start_rank + (group_sizes - 1) / 2
        rank_map = group_id.map(group_avg_rank)
    else:
        raise ValueError(f"method must be 'min' or 'average', got {method!r}")

    return rank_map.reindex(series.index)


assert np.all(
    fuzzy_rank(pd.Series([1, 2, 3, 5, 7, 11, 20], index=list("ABCDEFG")), 2)
    == pd.Series([1, 1, 1, 1, 1, 6, 7], index=list("ABCDEFG"))
)
assert np.all(
    fuzzy_rank(
        pd.Series([1, 2, 3, 5, 7, 11, 20], index=list("ABCDEFG")),
        tolerance=2,
        method="average",
    )
    == pd.Series([3, 3, 3, 3, 3, 6, 7], index=list("ABCDEFG"))
)

for site in ["marvin"]:
    for rank_fun in [
        {"func": fuzzy_rank, "tolerance": 1e-6, "method": "average"},
        {
            "func": fuzzy_rank,
            "tolerance": get_threshold(1 - 0.05),
            "method": "average",
        },
    ]:
        print(f"--- {site} ---")
        print(f"--- {rank_fun} ---")
        hm_data = df.query(f"~is_pysacess and site == '{site}'")
        # rank-wise, fx_best and optimality gap are equivalent
        hm_data["rank"] = hm_data.groupby("problem")["fx_best"].transform(
            **rank_fun
        )

        hm_data = hm_data.query(
            f"~is_pysacess and site == '{site}'"
        ).pivot_table(
            index=["optimizer_label"],
            columns="problem",
            values="rank",
            aggfunc="mean",
        )
        print("Shape", hm_data.shape)
        print("#NA values:", hm_data.isna().sum().sum())
        print("inf values:", np.isinf(hm_data).sum().sum())
        assert np.isinf(hm_data).sum().sum() == 0

        cmap = plt.colormaps.get_cmap("Greens_r").copy()
        cmap.set_bad("red")

        with plt.rc_context(
            {
                "font.size": 10,
                "xtick.labelsize": 10,
                "ytick.labelsize": 10,
                "axes.labelsize": 10,
            }
        ):
            plt.figure(figsize=(8, 8))

            # # successful optimizers per problem
            df_row_bar = hm_data.mean(axis=1)
            row_ha = HeatmapAnnotation(
                test=anno_barplot(
                    df_row_bar,
                    height=15,
                    colors="#008080",
                    label="avg. avg. rank",
                    legend=False,
                    ylim=(1, N_OPTIMIZERS * N_RUNS),
                ),
                axis=0,
                label_kws={
                    "rotation": 0,
                    "fontsize": 8,
                    "horizontalalignment": "center",
                    "verticalalignment": "bottom",
                },
            )

            # # solved problems per optimizer
            df_col_bar = hm_data.mean(axis=0)
            col_ha = HeatmapAnnotation(
                bla=anno_barplot(
                    df_col_bar,
                    height=15,
                    colors="#008080",
                    label="avg. avg. rank",
                    legend=False,
                ),
                axis=1,
                label_kws={
                    "rotation": 90,
                    "fontsize": 8,
                    "horizontalalignment": "center",
                },
            )
            cm = ClusterMapPlotter(
                data=(
                    hm_data
                    # order by number of solved
                    .loc[df_row_bar.sort_values(ascending=True).index, :]
                ),
                right_annotation=row_ha,
                col_split=problem_df.set_index(["short"]).loc[:].difficulty,
                col_split_gap=2,
                row_dendrogram=False,
                col_dendrogram=False,
                row_cluster=False,
                col_cluster=False,
                show_rownames=True,
                show_colnames=True,
                row_names_side="left",
                cmap=cmap,
                xticklabels_kws=dict(labelrotation=90),
                legend_kws=dict(
                    extend=None,
                ),
                label="Average rank",
                linecolor="white",
                linewidth=0.5,
                xlabel="Problem",
                ylabel="Optimisation method",
            )
            cm.ax_heatmap.set_aspect("equal")

            if hasattr(col_ha, "axes"):
                col_ha.axes.flatten()[0].yaxis.set_label_text(
                    col_ha.axes.flatten()[1].yaxis.get_label_text()
                )
                col_ha.axes.flatten()[1].yaxis.set_label_text("")
                col_ha.axes.flatten()[0].yaxis.label.set_visible(True)
                col_ha.axes.flatten()[0].yaxis.label.set_fontsize(
                    col_ha.axes.flatten()[1].yaxis.label.get_fontsize()
                )

            row_ha.axes.flatten()[0].xaxis.set_ticks_position("top")
            row_ha.axes.flatten()[0].xaxis.set_tick_params(labelrotation=90)

            plt.suptitle(f"{site} -- {rank_fun}")
            plt.savefig(
                f"out/fval_ranking{'_fuzzy_tol' + str(rank_fun['tolerance']) if rank_fun['func'] == fuzzy_rank else ''}.svg",
                bbox_inches="tight",
            )

## Reproducibility of optimality gaps between sites

Figure 3

In [ ]:
# for each optimizer x problem, plot optimality gap in marvin vs ft3
df_pivot = (
    df.query("~is_pysacess")
    .pivot_table(
        index=["problem", "optimizer_label", "problem_color"],
        columns="site",
        values="optimality_gap_site_without_pyscat",
        aggfunc="min",
    )
    .reset_index()
    .rename(columns={"ft3": "og_ft3", "marvin": "og_marvin"})
)


def plot_fx_best_correlation_all(ax: plt.Axes | None = None):
    """Scatter plot of fx_best across sites and problems and optimizers."""
    if ax is None:
        fig, ax = plt.subplots()

    x = df_pivot["og_ft3"] + log_offset
    y = df_pivot["og_marvin"] + log_offset
    ax.scatter(x, y, marker=".", alpha=0.5, color=df_pivot["problem_color"])
    ax.set_xscale("log")
    ax.set_yscale("log")

    ax.set_xlabel(f"Optimality gap +{log_offset} on FinisTerrae III")
    ax.set_ylabel(f"Optimality gap +{log_offset} on Marvin")

    problem_to_color = (
        df_pivot.set_index("problem")["problem_color"]
        .drop_duplicates()
        .to_dict()
    )
    handles = [
        Line2D(
            [],
            [],
            marker="o",
            linestyle="None",
            markerfacecolor=problem_to_color[p],
            markeredgecolor="w",
            label=p,
            alpha=0.5,
        )
        for p in problem_to_color
    ]
    ax.legend(
        handles=handles, title="Problem", ncol=1, bbox_to_anchor=(1.01, 1.02)
    )

    lims = np.array(
        [min(y.min(), x.min()), max(y.max(), x[np.isfinite(x)].max())]
    )
    lims_padded = (lims[0] / 10, lims[1] * 10)
    ax.set_xlim(*lims_padded)
    ax.set_ylim(*lims_padded)

    ax.set_aspect("equal", adjustable="box")
    # plot diagonal
    xmin, xmax = ax.get_xlim()
    ax.plot(
        [xmin / 10, xmax * 10],
        [xmin / 10, xmax * 10],
        color="k",
        lw=1,
        zorder=-1000000,
    )

    assert np.isfinite(df_pivot["og_ft3"]).all()
    assert np.isfinite(df_pivot["og_marvin"]).all()
    r2 = spearmanr(
        df_pivot["og_marvin"],
        df_pivot["og_ft3"],
    ).statistic
    ax.text(
        0.05,
        0.95,
        f"$\\rho_s$ = {r2:.3f}",
        transform=ax.transAxes,
        ha="left",
        va="top",
    )

    ticks = 10 ** np.arange(
        np.floor(np.log10(lims[0])), np.ceil(np.log10(lims[1]) / 3) * 3, step=3
    )
    ax.set_xticks(ticks)
    ax.set_yticks(ticks)


with plt.rc_context(
    rc={
        "figure.figsize": (3, 3),
        "figure.dpi": 300,
        "font.size": 6,
        "lines.markersize": 3,
    }
):
    plot_fx_best_correlation_all()

plt.savefig("out/Figure3D.svg")

In [ ]:
# spearman correlation per problem
problem_to_spearman = {}
for problem_id, group_df in df_pivot.groupby("problem"):
    x = group_df.og_ft3
    y = group_df.og_marvin

    problem_to_spearman[problem_id] = spearmanr(x, y).statistic

problem_spearman_df = (
    pd.DataFrame(
        {
            "problem": problem_to_spearman.keys(),
            "spearman": problem_to_spearman.values(),
        }
    )
    .set_index("problem")
    .sort_values(by="spearman", ascending=False)
)
assert problem_spearman_df.notna().all().all(), problem_spearman_df
problem_spearman_df

In [ ]:
problem_spearman_df.describe()

In [ ]:
# pick examples across observed correlation strengths
representative_examples = [
    problem_spearman_df.index[
        problem_spearman_df.spearman
        == problem_spearman_df.spearman.quantile(q, interpolation="nearest")
    ][0]
    for q in reversed((0, 1 / 3, 2 / 3, 1))
]
representative_examples

In [ ]:
# Correlation plots for optimality gap across sites (Fig 3C)


def plot_problem_correlation(
    problem_id: str,
    ax: plt.Axes | None = None,
    label_axes=False,
    label_outliers=False,
):
    """Plot optimality gap correlation across sites for one problem."""
    from matplotlib.ticker import LogLocator

    tmp_df = (
        df.query("~is_pysacess and problem == @problem_id")
        .pivot_table(
            index=["problem", "optimizer_label"],
            columns="site",
            values="optimality_gap_global_without_pyscat",
            aggfunc="min",
        )
        .reset_index()
        .rename(columns={"ft3": "og_ft3", "marvin": "og_marvin"})
    )
    assert len(tmp_df) == N_OPTIMIZERS, len(tmp_df)
    assert min(min(tmp_df.og_marvin), min(tmp_df.og_ft3)) == 0

    # ensure positive values for log axes
    x = tmp_df.og_ft3 + log_offset
    y = tmp_df.og_marvin + log_offset
    assert (x > 0).all()
    assert (y > 0).all()
    if ax is None:
        _, ax = plt.subplots()

    ax.scatter(x, y, marker=".")
    ax.set_xscale("log")
    ax.set_yscale("log")

    ax.set_box_aspect(1)

    if label_outliers:
        for xx, yy, optimizer in zip(
            tmp_df.og_ft3,
            tmp_df.og_marvin,
            tmp_df.optimizer_label,
            strict=False,
        ):
            if np.abs(xx - yy) > 10 or (
                xx != 0 and yy != 0 and np.abs(np.log10(xx / yy)) > 2
            ):
                ax.annotate(
                    optimizer,
                    (xx, yy),
                )

    if label_axes is not False:
        ax.set_xlabel(f"Optimality gap +{log_offset} on FinisTerrae III")
        ax.set_ylabel(f"Optimality gap +{log_offset} on Marvin")

    xmin, xmax = ax.get_xlim()
    ymin, ymax = ax.get_ylim()
    low = min(xmin, ymin)
    high = max(xmax, ymax)

    ax.set_xlim(low, high)
    ax.set_ylim(low, high)

    # diagonal
    ax.plot(
        [low, high],
        [low, high],
        color="k",
        linestyle="-",
        linewidth=0.5,
        zorder=-100,
    )

    # same (x,y) tick positions
    maj_locator = LogLocator(numticks=5)
    ax.yaxis.set_major_locator(maj_locator)
    ax.xaxis.set_major_locator(maj_locator)

    ax.set_title(problem_id)


def plot_corr_lines(
    ax: plt.Axes | None = None,
    representative_examples: Sequence[str] = None,
):
    """Correlation lines"""
    from matplotlib.patches import PathPatch
    from matplotlib.path import Path

    representative_examples = (
        representative_examples if representative_examples is not None else []
    )

    if ax is None:
        _, ax = plt.subplots()

    # no x-axis ticks
    ax.xaxis.set_major_locator(plt.NullLocator())
    # y-axis on the right
    ax.yaxis.tick_right()
    ax.yaxis.set_label_position("right")

    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)

    for i, (problem, spearman) in enumerate(
        zip(
            problem_spearman_df.index,
            problem_spearman_df.spearman,
            strict=False,
        )
    ):
        ax.hlines(
            y=spearman,
            xmin=0,
            xmax=1,
            linewidth=1,
        )
        text_x = -1
        text_y = 1 - i * (1 / len(problem_spearman_df))
        x_attach = 0
        y_attach = spearman
        ax.annotate(
            problem,
            xy=(x_attach, y_attach),
            xycoords="axes fraction",
            xytext=(text_x, text_y),
            textcoords="axes fraction",
            fontsize=plt.rcParams["ytick.labelsize"],
            fontweight=(
                "bold"
                if problem in representative_examples
                else plt.rcParams["font.weight"]
            ),
            ha="right",
            va="center",
        )

        # draw connecting lines
        #                 pt3 --- end
        #               /
        # start --- pt2
        pad = 0.02
        len_h = 0.1
        start = text_x + pad, text_y
        end = x_attach - pad, y_attach
        pt2 = start[0] + len_h, start[1]
        pt3 = end[0] - len_h, end[1]
        verts = [start, pt2, pt3, end]
        codes = [Path.MOVETO, Path.LINETO, Path.LINETO, Path.LINETO]
        patch = PathPatch(
            Path(verts, codes),
            transform=ax.transAxes,
            facecolor="none",
            linewidth=0.5,
        )
        ax.figure.add_artist(patch)

        if problem in representative_examples:
            # draw a line stub to the right of the plotting area for manual extension later on
            ax.axhline(
                spearman,
                1.5,
                2,
                clip_on=False,
                color="white",
                linewidth=1,
                zorder=-10,
            )


def plot_fig3c():
    fig, axs = plt.subplot_mosaic(
        "A.BC\nA.DE",
        layout="constrained",
        width_ratios=[0.1, 0.06, 0.35, 0.35],
    )

    plot_corr_lines(axs["A"], representative_examples=representative_examples)

    # show detailed correlation plot for representative examples
    for problem_id, ax in zip(
        representative_examples, list(axs.values())[1:], strict=True
    ):
        plot_problem_correlation(problem_id, ax=ax)
        ax.text(
            0.02,
            0.98,
            f"$\\rho_S={problem_spearman_df.loc[problem_id].spearman:.2f}$",
            transform=ax.transAxes,
            ha="left",
            va="top",
        )

    # super x labels between axes
    fig.text(
        0.39,
        0.5,
        f"Optimality gap +{log_offset} on Marvin",
        fontsize=plt.rcParams["figure.titlesize"],
        ha="center",
        va="center",
        rotation=90,
    )
    fig.text(
        0.73,
        -0.03,
        f"Optimality gap +{log_offset} on FinisTerrae III",
        fontsize=plt.rcParams["figure.titlesize"],
        ha="center",
        va="center",
    )


with plt.rc_context(
    {
        "figure.figsize": (5, 3),
        "font.size": 6,
        "figure.dpi": 300,
    }
):
    plot_fig3c()
    plt.savefig("out/Figure3C.pdf", bbox_inches="tight")
    plt.savefig("out/Figure3C.svg", bbox_inches="tight")

### Statistical tests

In [ ]:
# Mann-Whitney U rank test on two independent samples
# distribution of best value per run across the 10 runs

df_pivot = (
    df.query("~is_pysacess")
    .pivot_table(
        index=["problem", "optimizer_label", "run_idx"],
        columns="site",
        # values="optimality_gap_site_without_pyscat",
        values="optimality_gap_global_without_pyscat",
    )
    .reset_index()
    .rename(columns={"ft3": "og_ft3", "marvin": "og_marvin"})
)

rows = []
for (problem, optimizer), grouped in df_pivot.groupby(
    ["problem", "optimizer_label"]
):
    x, y = grouped["og_ft3"], grouped["og_marvin"]
    _, p_two_sided = mannwhitneyu(x, y, alternative="two-sided")
    _, p_ft3_less = mannwhitneyu(x, y, alternative="less")
    _, p_ft3_greater = mannwhitneyu(x, y, alternative="greater")
    rows.append(
        {
            "Problem": problem,
            "Optimizer": optimizer,
            "p_two_sided": p_two_sided,
            "p_ft3_less": p_ft3_less,
            "p_ft3_greater": p_ft3_greater,
            "og_min_ft3": x.min(),
            "og_min_marvin": y.min(),
        }
    )

df_p = pd.DataFrame(rows)
df_p

In [ ]:
for alpha in [0.01, 0.05]:
    n = (df_p.p_two_sided < alpha).sum()
    print(
        f"Combinations with p < {alpha}: {n}/{len(df_p)} ({n / len(df_p):.2%})"
    )

In [ ]:
df_p_pivot = df_p.pivot_table(
    values="p_two_sided", index="Problem", columns="Optimizer"
)
assert np.isfinite(df_p_pivot).all().all()

with (
    plt.rc_context(
        rc={
            "figure.figsize": (8, 10),
        }
    ),
    sns.plotting_context("paper", font_scale=1.2),
):
    cmap = plt.colormaps.get_cmap("viridis").copy()
    cmap.set_over("white")
    cmap.set_bad("red")

    fig, ax = plt.subplots()
    norm = mpl.colors.Normalize(vmin=0.0, vmax=0.01, clip=False)

    sns.heatmap(
        df_p_pivot,
        cmap=cmap,
        norm=norm,
        square=True,
        linecolor="white",
        linewidth=1,
        cbar_kws={"label": "p-value", "shrink": 0.5, "extend": "max"},
        ax=ax,
    )
    ax.set_xticks(0.5 + np.arange(df_p_pivot.shape[1]))
    ax.set_yticks(0.5 + np.arange(df_p_pivot.shape[0]))
    ax.set_xticklabels(df_p_pivot.columns, rotation=90)
    ax.set_yticklabels(df_p_pivot.index, rotation=0)
    ax.set_xlabel("Optimisation method")

    cbar = ax.collections[0].colorbar
    cbar.ax.spines[["top", "bottom", "left", "right"]].set_visible(True)
    for patch in cbar._extend_patches:
        patch.set_edgecolor("black")
        patch.set_linewidth(cbar.ax.spines["left"].get_linewidth())
        patch.set_antialiased(True)

    # draw box around heatmap
    for side in ["left", "right", "top", "bottom"]:
        spine = ax.spines[side]
        spine.set_visible(True)
        spine.set_linewidth(1.5)
        spine.set_edgecolor("black")

    plt.savefig("out/Figure3E.svg", bbox_inches="tight")

In [ ]:
print(
    f"Number of cells<0.01: {df_p_pivot.lt(0.01).sum().sum()} ({df_p_pivot.lt(0.01).sum().sum() / (df_p_pivot.shape[0] * df_p_pivot.shape[1]):%})"
)

In [ ]:
df_p_pivot.lt(0.01).sum(axis=0).rename("num_significant").sort_values(
    ascending=False
)

In [ ]:
df_p_pivot.lt(0.01).sum(axis=1).rename("num_significant").sort_values(
    ascending=False
)

In [ ]:
with plt.rc_context(
    rc={
        "figure.figsize": (2, 3.5),
        "figure.dpi": 300,
        "font.size": 6,
    }
):
    fig, (ax1, ax2) = plt.subplots(nrows=2, ncols=1, layout="constrained")

    ax = ax1
    sns.histplot(df_p.p_two_sided, binwidth=0.05, shrink=0.8, ax=ax)
    ax.set_xlabel("p-value")
    ax.set_ylabel("Count")
    ax.set_title("Differences (two-sided)")

    df_tmp = (
        df_p[["Problem", "Optimizer", "p_ft3_greater", "p_ft3_less"]]
        .rename(
            columns={
                "p_ft3_greater": "Marvin better",
                "p_ft3_less": "FT III better",
            }
        )
        .melt(
            id_vars=["Problem", "Optimizer"],
        )
        .rename(columns={"variable": "Condition", "value": "p-value"})
    )
    assert len(df_tmp) == N_OPTIMIZERS * N_PROBLEMS * 2

    ax = ax2
    g = sns.histplot(
        data=df_tmp,
        x="p-value",
        hue="Condition",
        multiple="dodge",
        binwidth=0.1,
        common_norm=False,
        alpha=0.8,
        shrink=0.8,
        ax=ax,
    )
    ax.set_title("Objective function value\nlarger or smaller (one-sided)")

    plt.savefig("out/Figure3E2.svg")

In [ ]:
alphas = np.linspace(0, 1, 21)
pos_tests = np.array(
    [sum(df_p.p_two_sided < alpha) / len(df_p) for alpha in alphas]
)

with plt.rc_context(
    rc={
        "figure.figsize": (2.5, 2.5),
        "figure.dpi": 300,
        "font.size": 6,
        "lines.markersize": 3,
    }
):
    fig, ax = plt.subplots()
    ax.set_aspect("equal", adjustable="box")
    ax.plot([0, 1], [0, 1], "--", color="grey", lw=1)
    ax.plot(alphas, pos_tests, "o-", c="k", markeredgecolor="w")
    ax.set_xlabel("Significance level")
    ax.set_ylabel("Fraction of positive tests")
    ax.set_xticks(np.linspace(0, 1, 6))
    ax.set_yticks(np.linspace(0, 1, 6))

    plt.savefig("out/Figure3F.svg")

## Convergence curves

Trajectories of optimality per-run.

In [ ]:
def convergence_plot(
    path: Path,
    problem: str,
    ax: plt.Axes | None = None,
    xmin=1e2,
    xmax=1e4,
    reference_color="r",
    run_color="grey",
    best_run_color="b",
) -> plt.Axes:
    """
    Plot convergence curves of the 10 different runs.

    Plot the global best fval, highlight the best run, and show the optimality gap.

    :param path: Path to the folder where the results are saved.
    :param problem: Short name of the problem.
    :param ax: Axes to plot on
    :param xmin: Minimum x-axis value
    :param xmax: Maximum x-axis value.
        NOTE: This is currently assumed to be the walltime limit. Any better fvals at t > xmax would be ignored.
    :param reference_color: Color for the global best fval
    :param run_color: Line color for the runs
    :param best_run_color: Color for highlighting the best run
    """
    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 4), layout="constrained")

    global_best_fx: float = df_best_fvals.loc[problem, "without_pyscat"]

    def plot_reference(ax: plt.Axes):
        ax.axhline(
            global_best_fx, color=reference_color, linestyle="--", zorder=11
        )

    plot_reference(ax)

    dfs = [
        pd.read_parquet(f, columns=["time", "fval"])
        # truncate at the walltime limit
        .query(f"time <= {xmax}")
        for f in path.glob("*.parquet")
    ]
    # extend lines to the walltime limit
    # (assumes that the trajectories have been truncated at the walltime limit!)
    dfs = [
        pd.concat(
            [df, pd.DataFrame([dict(time=xmax, fval=df.fval.min())])],
            ignore_index=True,
        )
        for df in dfs
    ]
    ymax = global_best_fx
    endpoint_to_line = {}
    for df in dfs:
        ymax: float = max(ymax, df.query("@xmin <= time <= @xmax").fval.max())
        lines = ax.step(df.time, df.fval, where="post", c=run_color)
        endpoint_to_line[df.fval.min()] = lines[0]

    local_best_fx = min(endpoint_to_line.keys())
    optimality_gap = local_best_fx - global_best_fx
    # highlight curve with the minimum value
    best_line = endpoint_to_line[local_best_fx]
    best_line.set_color(best_run_color)

    ax.set_xlim(xmin, xmax)
    yrange = ymax - global_best_fx
    ax.set_ylim(global_best_fx - yrange * 0.05, ymax)
    ax.set_xlabel("Wall time [s]")
    ax.set_ylabel("Objective function")
    ax.set_xscale("log")
    ax.set_box_aspect(1)

    axins = ax.inset_axes((1.2, 0.0, 0.5, 0.5), transform=ax.transAxes)
    for df in dfs:
        lines = axins.step(df.time, df.fval, where="post", c=run_color)
        endpoint_to_line[df.fval.min()] = lines[0]
    best_line = endpoint_to_line[min(endpoint_to_line.keys())]
    best_line.set_color(best_run_color)
    best_line.set_zorder(10)
    plot_reference(axins)

    axins.set_xscale("log")
    if optimality_gap > 0:
        padding_bottom = max(1, 0.15 * optimality_gap)
        padding_top = max(5, 0.15 * optimality_gap)
        axins.set_ylim(
            global_best_fx - padding_bottom, local_best_fx + padding_top
        )
    else:
        # og==0, global_best_fx==local_best_fx, choose non-zero y-range
        import math

        axins.set_ylim(
            math.floor(global_best_fx - 1), math.ceil(local_best_fx + 1)
        )
    # zoom box height as ax fraction
    _, tmp1 = ax.transLimits.transform((0, axins.get_ylim()[0]))
    _, tmp2 = ax.transLimits.transform((0, axins.get_ylim()[1]))
    height_ax_frac = tmp2 - tmp1
    # use the same fraction of the x-axis

    axins.set_xlim(
        ax.transScale.inverted().transform(
            ax.transLimits.inverted().transform((1 - height_ax_frac, 1))
        )[0],
        xmax,
    )
    axins.set_title(f"optimality gap: {optimality_gap:.3g}")

    # annotate best local and best global lines
    axins.annotate(
        f"{local_best_fx:.2f}",
        xy=(axins.get_xlim()[0], local_best_fx),
        horizontalalignment="left",
        verticalalignment="bottom",
        xytext=(5, 1),
        textcoords="offset points",
        c=best_line.get_color(),
    )
    axins.annotate(
        f"{global_best_fx:.2f}",
        xy=(axins.get_xlim()[0], global_best_fx),
        horizontalalignment="left",
        verticalalignment="top",
        xytext=(5, -2),
        textcoords="offset points",
        c=reference_color,
    )
    plt.setp(axins.get_xticklabels(which="both"), rotation=30, ha="right")
    ax.indicate_inset_zoom(axins, edgecolor="blue", linewidth=2, alpha=0.2)

    return ax

In [ ]:
selected_examples = (
    ("Elowitz", "SLSQP"),
    ("Elowitz", "BFGS"),
    ("Elowitz", "pyswarm"),
)

if Path("data/data_samples").is_dir():
    with plt.rc_context(
        rc={
            "figure.figsize": (3, 6),
            "figure.dpi": 300,
            "font.size": 6,
            "lines.markersize": 3,
        }
    ):
        fig, axs = plt.subplots(3, 1, layout="constrained", sharex=True)
        for (problem, optimizer), ax in zip(
            selected_examples, axs, strict=False
        ):
            ax = convergence_plot(
                Path(
                    f"data_samples/per_run_traj_problem={problem}/optimizer={optimizer}/"
                ),
                problem=problem,
                xmax=RELATIVE_WALLTIME_LIMIT
                * problem_df.set_index("short").loc[problem, "walltime_s"],
                ax=ax,
            )
            ax.set_title(
                f"{optimizer_df.set_index('output_dir').loc[optimizer, 'optimizer_label']} - {problem}"
            )

        plt.savefig("out/Figure4B.svg")

## Heatmap of optimality gaps

In [ ]:
def my_hist(
    data: np.ndarray,
    upper_bound,
    base_cmap=plt.colormaps.get_cmap("Greens_r"),
    overflow_color=np.array([[1, 0, 0, 1]]),
    edgecolor=None,
    figsize=(3, 2),
):
    normal_edges = np.linspace(data.min(), upper_bound, 30)
    bin_width = normal_edges[1] - normal_edges[0]
    overflow_max = upper_bound + bin_width

    plot_data = np.minimum(data, overflow_max - 1e-9)
    bins = np.r_[normal_edges, overflow_max]

    fig, ax = plt.subplots(figsize=figsize)

    counts, edges, patches = ax.hist(plot_data, bins=bins, edgecolor=edgecolor)

    normal_colors = base_cmap(np.linspace(0, 1, len(patches) - 1))
    colors = np.vstack([normal_colors, overflow_color])

    for patch, color in zip(patches, colors, strict=False):
        patch.set_facecolor(color)

    cmap = ListedColormap(colors)
    norm = BoundaryNorm(edges, cmap.N)

    sm = ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])

    cbar = fig.colorbar(
        sm,
        ax=ax,
        orientation="horizontal",
        pad=0.01,
        boundaries=edges,
        spacing="proportional",
    )

    # Regular continuous ticks
    regular_ticks = np.arange(data.min(), upper_bound, step=0.5)
    # Add one extra tick centered in overflow bin
    overflow_tick = upper_bound + bin_width / 2

    cbar.set_ticks(np.r_[regular_ticks, overflow_tick])
    cbar.set_ticklabels(
        [str(t) for t in regular_ticks] + [f">{upper_bound:.2f}"]
    )
    cbar.minorticks_off()

    ax.set_xlim(edges[0], edges[-1])
    cbar.ax.set_xlim(edges[0], edges[-1])
    ax.xaxis.set_visible(False)

    cbar.ax.set_xlabel("OG")
    ax.yaxis.set_visible(False)

    # secondary alpha (significance level) axis
    secax = cbar.ax.secondary_xaxis("bottom")
    secax.set_xlabel("$\\alpha$")
    alphas = [
        a
        for a in (0.01, 0.05, 0.1, 0.2, 0.3, 0.5, 1.0)
        if get_threshold(1 - a) <= upper_bound
    ]
    alpha_labels = [f"{a:.2f}".rstrip("0").rstrip(".") for a in alphas]
    alpha_labels[0] = ">" + alpha_labels[0]
    secax.set_xticks([get_threshold(1 - a) for a in alphas])
    secax.set_xticklabels(alpha_labels)
    secax.spines["bottom"].set_position(("outward", 40))

    cbar.ax.tick_params(which="minor", length=3)
    cbar.ax.tick_params(which="major", length=6)

    for spine in ax.spines.values():
        spine.set_visible(False)


cell_highlight_color = "#FF00FF"

for alpha in [0.05]:
    print(f"alpha={alpha}")
    for site in ["marvin"]:
        print(f"--- {site} ---")
        # minimal optimality gap heatmap for each problem/optimizer
        hm_data = df.query(f"site == '{site}' and ~is_pysacess").pivot_table(
            index="optimizer_label",
            columns="problem",
            values="optimality_gap_site_without_pyscat",
            aggfunc="min",
            fill_value=np.nan,
        )
        print("Shape", hm_data.shape)
        print("#NA values:", hm_data.isna().sum().sum())
        print("inf values:", np.isinf(hm_data).sum().sum())
        assert np.isinf(hm_data).sum().sum() == 0
        print("OG range:", hm_data.min().min(), "to", hm_data.max().max())

        # clip at chi²-based optimality gap threshold
        cap = get_threshold(percentile=1 - alpha)
        print(f"Capping at {cap}")
        clipped = (
            hm_data.clip(upper=cap) if cap is not None else hm_data.copy()
        )
        original_max = np.nanmax(hm_data.values)
        clipped_max = np.nanmax(clipped.values)
        cap_applied = (cap is not None) and (original_max > cap)

        # replace NA for clustering
        filled = clipped.fillna(clipped.max().max() + 1e-3)

        solved_df = hm_data < cap
        print(solved_df.sum(axis=0).describe())
        print(solved_df.sum(axis=1).describe())

        cmap = plt.colormaps.get_cmap("Greens_r").copy()
        cmap.set_over("white")
        cmap.set_bad("red")

        with plt.rc_context(
            {
                "font.size": 10,
                "xtick.labelsize": 10,
                "ytick.labelsize": 10,
                "axes.labelsize": 10,
            }
        ):
            plt.figure(figsize=(8, 8))

            # # successful optimizers per problem
            df_row_bar = solved_df.sum(axis=1)
            row_ha = HeatmapAnnotation(
                test=anno_barplot(
                    df_row_bar,
                    height=15,
                    colors="#008080",
                    label="# Solved\nproblems",
                    legend=False,
                    ylim=(0, N_PROBLEMS),
                ),
                axis=0,
                label_kws={
                    "rotation": 0,
                    "fontsize": 8,
                    "horizontalalignment": "center",
                    "verticalalignment": "bottom",
                },
            )

            # solved problems per optimizer
            df_col_bar = solved_df.sum(axis=0)
            col_ha = HeatmapAnnotation(
                bla=anno_barplot(
                    df_col_bar,
                    height=15,
                    colors="#008080",
                    label="# Successful\noptimisers",
                    legend=False,
                    ylim=(0, N_OPTIMIZERS),
                ),
                axis=1,
                label_kws={
                    "rotation": 90,
                    "fontsize": 8,
                    "horizontalalignment": "center",
                },
            )
            cm = ClusterMapPlotter(
                data=(
                    # some offset for the non-solved pairs to ensure they are outside the color scale
                    (filled + 100 * ~solved_df)
                    # order by number of solved
                    .loc[
                        df_row_bar.sort_values(ascending=False).index,
                        df_col_bar.sort_values(ascending=False).index,
                    ]
                ),
                top_annotation=col_ha,
                right_annotation=row_ha,
                col_split=problem_df.set_index(["short"])
                .loc[df_col_bar.sort_values(ascending=False).index]
                .difficulty,
                col_split_gap=2,
                row_dendrogram=False,
                col_dendrogram=False,
                row_cluster=False,
                col_cluster=False,
                show_rownames=True,
                show_colnames=True,
                row_names_side="left",
                cmap=cmap,
                xticklabels_kws=dict(labelrotation=90),
                # color bar
                legend_kws=dict(
                    extend="max",
                    extendfrac=0.1,
                ),
                vmin=0,
                vmax=cap,
                label="Optimality gap",
                linecolor="white",
                linewidth=0.5,
                xlabel="Problem",
                ylabel="Optimisation method",
            )
            cm.ax_heatmap.set_aspect("equal")

            col_ha.axes.flatten()[0].yaxis.set_label_text(
                col_ha.axes.flatten()[1].yaxis.get_label_text()
            )
            col_ha.axes.flatten()[1].yaxis.set_label_text("")
            col_ha.axes.flatten()[0].yaxis.label.set_visible(True)
            col_ha.axes.flatten()[0].yaxis.label.set_fontsize(
                col_ha.axes.flatten()[1].yaxis.label.get_fontsize()
            )

            row_ha.axes.flatten()[0].xaxis.set_ticks_position("top")
            row_ha.axes.flatten()[0].xaxis.set_tick_params(labelrotation=0)

            # highlight selected examples
            if True and selected_examples:
                # the heatmap cells are in axes[6], not in ax_heatmap ...
                ax_real = cm.ax_heatmap.figure.axes[6]

                row_labels = cm.row_order[0]
                col_labels = cm.col_order[0]
                for problem, optimizer in selected_examples:
                    row_idx = row_labels.index(
                        optimizer_df.set_index("output_dir").loc[
                            optimizer, "optimizer_label"
                        ]
                    )
                    col_idx = col_labels.index(problem)

                    rect = patches.Rectangle(
                        (col_idx, row_idx),
                        width=1,
                        height=1,
                        zorder=999,
                        edgecolor=cell_highlight_color,
                        facecolor="none",
                        clip_on=False,
                    )
                    ax_real.add_patch(rect)

            plt.suptitle(site)
            plt.savefig(f"out/Figure4C_{site}.svg", bbox_inches="tight")

            my_hist(
                data=hm_data.values.flatten(),
                upper_bound=cap,
                overflow_color=np.array([[1, 1, 1, 1]]),
                edgecolor="black",
                figsize=(2, 1),
            )
            plt.savefig(f"out/Figure4C_cbar_{site}.svg", bbox_inches="tight")

In [ ]:
# Some stats:
site = "marvin"
alpha = 0.05
tmp_df = (
    df.query(f"site == '{site}' and ~is_pysacess")
    .pivot_table(
        index="optimizer_label",
        columns="problem",
        values="optimality_gap_site_without_pyscat",
        aggfunc="min",
        fill_value=np.nan,
    )
    .lt(get_threshold(1 - alpha))
)


pd.DataFrame(
    {
        "solved_problems_by_optimizer_abs": tmp_df.sum(axis=1),
        "solved_problems_by_optimizer_rel": tmp_df.sum(axis=1)
        / tmp_df.shape[1],
    }
).describe()

In [ ]:
tmp_df.sum(axis=1).sort_values(ascending=False)

In [ ]:
pd.DataFrame(
    {
        "solved_problems_by_problem_abs": tmp_df.sum(axis=0),
        "solved_problems_by_problem_rel": tmp_df.sum(axis=0) / tmp_df.shape[0],
    }
).describe()

In [ ]:
print(tmp_df.sum(axis=0).sort_values())

## Optimality gaps aggregated by problem / optimizer

In [ ]:
df_tmp = (
    df.query("site == 'marvin' and ~is_pysacess")
    .groupby(["problem", "optimizer", "optimizer_label"])
    .min()
)


def add_split_spines(ax, gap_idx, xmax=None):
    lw = ax.spines["left"].get_linewidth()
    ec = ax.spines["left"].get_edgecolor()

    for spine in ax.spines.values():
        spine.set_visible(False)

    ymin, ymax = ax.get_ylim()
    xmin, xmax_ = ax.get_xlim()
    if xmax is None:
        xmax = xmax_

    # categories are at integer x; gap at gap_idx means left group ends at
    # gap_idx - 0.5 and right group starts at gap_idx + 0.5
    x_split_l = gap_idx - 0.5
    x_split_r = gap_idx + 0.5

    kw = dict(
        transform=ax.transData,
        color=ec,
        linewidth=lw,
        clip_on=False,
        solid_capstyle="butt",
    )

    for x0, x1 in [(xmin, x_split_l), (x_split_r, xmax)]:
        ax.add_line(Line2D([x0, x1], [ymax, ymax], **kw))  # top
        ax.add_line(Line2D([x0, x1], [ymin, ymin], **kw))  # bottom
        ax.add_line(Line2D([x0, x0], [ymin, ymax], **kw))  # left
        ax.add_line(Line2D([x1, x1], [ymin, ymax], **kw))  # right


def make_log10_yticks(ax: plt.Axes):
    ymin, ymax = ax.get_ylim()
    ymin, ymax = int(np.floor(ymin)), int(np.ceil(ymax)) + 1
    step = 1 if ymax - ymin < 10 else 3
    powers = np.arange(ymin, ymax, step)

    ax.set_yticks(list(powers))
    ax.set_yticklabels([f"$10^{{{p}}}$" for p in powers])

    # minor ticks at every power of 10 (no labels)
    all_powers = np.arange(ymin, ymax, 1)
    ax.yaxis.set_minor_locator(FixedLocator(all_powers))

In [ ]:
df_tmp = (
    df.query("site == 'marvin' and ~is_pysacess")
    .groupby(["problem", "optimizer", "optimizer_label"])
    .min()
)

with plt.rc_context(
    {
        "figure.dpi": 300,
        "font.size": 4.5,
        "lines.markersize": 1,
    }
):
    size = 3
    fig, (ax1, ax2) = plt.subplots(
        nrows=1,
        ncols=2,
        figsize=(17.5 / 2.54, 3),
        # sharey=True,
        width_ratios=(N_PROBLEMS + 1, N_OPTIMIZERS),
        layout="constrained",
    )

    # Median minimal optimality gap over problems
    ax = ax1
    order = (
        df_tmp.groupby(["problem", "difficulty"])
        .agg({"optimality_gap_site_without_pyscat_log10_offset": "median"})
        .sort_values("optimality_gap_site_without_pyscat_log10_offset")
        .reset_index()
        .set_index("problem")
    )
    # insert placeholder between easy and hard
    # (and extra items for padding so we get the same bar width as above)
    order = (
        order.query("difficulty == 'easy'").index.tolist()
        + [""]
        + order.query("difficulty != 'easy'").index.tolist()
        + ["", ""]
    )
    gap_idx = order.index("")

    sns.stripplot(
        data=df_tmp,
        x="problem",
        y="optimality_gap_site_without_pyscat_log10_offset",
        ax=ax,
        order=order,
        size=size,
    )
    sns.pointplot(
        data=df_tmp,
        x="problem",
        y="optimality_gap_site_without_pyscat_log10_offset",
        ax=ax,
        order=order,
        estimator="median",
        errorbar=None,
        markers="_",
        markeredgewidth=1,
        color="black",
        linestyle="none",
        zorder=10,
    )
    ax.hlines(
        y=np.log10(get_threshold(percentile=1 - 0.05, df=1) + log_offset),
        xmin=-0.5,
        xmax=gap_idx - 0.5,
        color="orange",
        zorder=-1,
    )
    ax.hlines(
        y=np.log10(get_threshold(percentile=1 - 0.05, df=1) + log_offset),
        xmin=gap_idx + 0.5,
        xmax=ax.get_xlim()[1] - 1.5,
        color="orange",
        zorder=-1,
    )
    make_log10_yticks(ax)

    ax.tick_params(axis="x", labelrotation=90)
    ax.set_xlabel("Problem")
    ax.set_ylabel(f"Optimality gap + {log_offset}")

    # hide tick at placeholder x
    tick = ax.xaxis.get_major_ticks()[gap_idx]
    tick.tick1line.set_visible(False)
    tick.tick2line.set_visible(False)

    ax.set_xlim(-0.5, len(order) - 0.5)
    add_split_spines(ax, gap_idx, xmax=len(order) - 2.5)

    # Median minimal optimality gap over optimizers
    ax = ax2
    order = (
        df_tmp.groupby("optimizer_label")
        .agg({"optimality_gap_site_without_pyscat_log10_offset": "median"})
        .sort_values("optimality_gap_site_without_pyscat_log10_offset")
        .index.values
    )
    sns.stripplot(
        data=df_tmp,
        x="optimizer_label",
        y="optimality_gap_site_without_pyscat_log10_offset",
        ax=ax,
        order=order,
        size=size,
    )
    sns.pointplot(
        data=df_tmp,
        x="optimizer_label",
        y="optimality_gap_site_without_pyscat_log10_offset",
        ax=ax,
        order=order,
        estimator="median",
        errorbar=None,
        markers="_",
        markeredgewidth=1,
        color="black",
        linestyle="none",
        zorder=10,
    )
    make_log10_yticks(ax)

    # add_threshold_hline
    ax.axhline(
        y=np.log10(get_threshold(percentile=1 - 0.05, df=1) + log_offset),
        color="orange",
        zorder=-1,
    )

    ax.tick_params(axis="x", labelrotation=90)
    ax.set_xlabel("Optimisation method")
    ax.set_ylabel(f"Optimality gap + {log_offset}")
    fig.align_xlabels()
    plt.savefig("out/Figure4DE.svg")
    plt.show()

## Heatmap of optimality gaps at different timepoints

In [ ]:
df_time_fracs = pd.read_csv("../data/fval_at_timepoints.csv")
df_time_fracs = df_time_fracs.join(
    problem_df.set_index(["short"]),
    on="problem",
    how="left",
    validate="many_to_one",
)
df_time_fracs = df_time_fracs.join(
    optimizer_df.set_index(["output_dir"]),
    on="optimizer",
    how="left",
    validate="many_to_one",
)
df_time_fracs = df_time_fracs.query("~excluded")

df_time_fracs["ref_site_without_pyscat"] = df_time_fracs.problem.map(
    site_best_df_without_pyscat.reset_index()
    .query("site == 'marvin'")
    .set_index("problem")["fx_best"]
)
df_time_fracs["optimality_gap_site_without_pyscat"] = (
    df_time_fracs["fval"] - df_time_fracs["ref_site_without_pyscat"]
)
df_time_fracs

In [ ]:
df_time_fracs.timepoint_fraction.value_counts()

In [ ]:
for frac in df_time_fracs.timepoint_fraction.unique():
    print(
        f"{frac}: {frac * problem_df.walltime_s.max() * RELATIVE_WALLTIME_LIMIT}s walltime for hard problems"
    )

print()
for frac in df_time_fracs.timepoint_fraction.unique():
    print(
        f"{frac}: {frac * problem_df.walltime_s.min() * RELATIVE_WALLTIME_LIMIT}s walltime for easy problems"
    )

In [ ]:
df_time_fracs.query("optimizer == 'nlopt_28' and problem == 'Zheng'")

In [ ]:
assert (
    df_time_fracs.groupby(["problem", "optimizer", "timepoint_fraction"])
    .count()
    .shape[0]
    / N_PROBLEMS
    / N_OPTIMIZERS
    == df_time_fracs.timepoint_fraction.nunique()
)

In [ ]:
assert np.all(
    df_time_fracs.groupby(
        ["problem", "optimizer", "timepoint_fraction"]
    ).count()["run_idx"]
    == 10
), (
    df_time_fracs.groupby(["problem", "optimizer", "timepoint_fraction"])
    .count()
    .query("run_idx != 10")
)

In [ ]:
df_time_fracs.timepoint_fraction.unique().tolist()

In [ ]:
nfinal_solved = df_time_fracs.query(
    f"timepoint_fraction=={df_time_fracs.timepoint_fraction.max()} and ~is_pysacess"
).pivot_table(
    index="optimizer_label",
    columns="problem",
    values="optimality_gap_site_without_pyscat",
    aggfunc="min",
    fill_value=np.nan,
) < get_threshold(1 - alpha)
problem_order = nfinal_solved.sum(axis=0).sort_values(ascending=False).index
optimizer_order = nfinal_solved.sum(axis=1).sort_values(ascending=False).index
col_split = problem_df.set_index(["short"]).loc[problem_order].difficulty
alpha = 0.05


with plt.rc_context(
    {
        "font.size": 10,
        "xtick.labelsize": 10,
        "ytick.labelsize": 10,
        "axes.labelsize": 10,
        "figure.titlesize": 18,
    }
):
    time_fracs = [0.001, 0.01, 0.1, 1.0]

    n_heatmaps = len(time_fracs)
    fig = plt.figure(figsize=(6.5 * n_heatmaps, 7))
    subfigs = fig.subfigures(nrows=1, ncols=n_heatmaps, wspace=0)

    for i_heatmap, (time_fraction, subfig) in enumerate(
        zip(time_fracs, subfigs, strict=False)
    ):
        print(f"alpha={alpha} time_fraction={time_fraction}")

        ax = subfig.subplots()
        plt.sca(ax)

        # minimal optimality gap heatmap for each problem/optimizer
        hm_data = df_time_fracs.query(
            f"timepoint_fraction=={time_fraction} and ~is_pysacess"
        ).pivot_table(
            index="optimizer_label",
            columns="problem",
            values="optimality_gap_site_without_pyscat",
            aggfunc="min",
            fill_value=np.nan,
        )
        print("Shape", hm_data.shape)
        print("#NA values:", hm_data.isna().sum().sum())
        print("inf values:", np.isinf(hm_data).sum().sum())
        assert np.isinf(hm_data).sum().sum() == 0
        print("OG range:", hm_data.min().min(), "to", hm_data.max().max())

        # clip at chi²-based optimality gap threshold
        cap = get_threshold(percentile=1 - alpha)
        print(f"Capping at {cap}")
        clipped = (
            hm_data.clip(upper=cap) if cap is not None else hm_data.copy()
        )
        original_max = np.nanmax(hm_data.values)
        clipped_max = np.nanmax(clipped.values)
        cap_applied = (cap is not None) and (original_max > cap)

        # replace NA for clustering
        filled = clipped.fillna(clipped.max().max() + 1e-3)

        solved_df = hm_data < cap

        cmap = plt.colormaps.get_cmap("Greens_r").copy()
        cmap.set_over("white")
        cmap.set_bad("red")

        # # successful optimizers per problem
        df_row_bar = solved_df.sum(axis=1)
        row_ha = HeatmapAnnotation(
            test=anno_barplot(
                df_row_bar,
                height=15,
                colors="#008080",
                label="# Solved\nproblems",
                legend=False,
                ylim=(0, N_PROBLEMS),
            ),
            axis=0,
            label_kws={
                "rotation": 0,
                "fontsize": 8,
                "horizontalalignment": "center",
                "verticalalignment": "bottom",
            },
        )

        # # solved problems per optimizer
        df_col_bar = solved_df.sum(axis=0)
        col_ha = HeatmapAnnotation(
            bla=anno_barplot(
                df_col_bar,
                height=15,
                colors="#008080",
                label="# Successful\noptimisers",
                legend=False,
                ylim=(0, N_OPTIMIZERS),
            ),
            axis=1,
            label_kws={
                "rotation": 90,
                "fontsize": 8,
                "horizontalalignment": "center",
            },
        )
        cm = ClusterMapPlotter(
            data=(
                # some offset for the non-solved pairs to ensure they are outside the color scale
                (filled + 100 * ~solved_df)
                # order by number of solved
                .loc[optimizer_order, problem_order]
            ),
            col_split=col_split,
            col_split_gap=2,
            row_dendrogram=False,
            col_dendrogram=False,
            row_cluster=False,
            col_cluster=False,
            show_rownames=i_heatmap == 0,
            show_colnames=True,
            row_names_side="left",
            cmap=cmap,
            legend=True,
            xticklabels_kws=dict(labelrotation=90),
            # color bar
            legend_kws=dict(
                extend="max",
                extendfrac=0.1,
            ),
            vmin=0,
            vmax=cap,
            label="Optimality gap",
            linecolor="white",
            linewidth=0.5,
            xlabel="Problem",
            ylabel="Optimisation method",
        )
        cm.ax_heatmap.set_aspect("equal")

        if hasattr(col_ha, "axes"):
            col_ha.axes.flatten()[0].yaxis.set_label_text(
                col_ha.axes.flatten()[1].yaxis.get_label_text()
            )
            col_ha.axes.flatten()[1].yaxis.set_label_text("")
            col_ha.axes.flatten()[0].yaxis.label.set_visible(True)
            col_ha.axes.flatten()[0].yaxis.label.set_fontsize(
                col_ha.axes.flatten()[1].yaxis.label.get_fontsize()
            )
        if hasattr(row_ha, "axes"):
            row_ha.axes.flatten()[0].xaxis.set_ticks_position("top")
            row_ha.axes.flatten()[0].xaxis.set_tick_params(labelrotation=0)

        subfig.suptitle(f"$t = {time_fraction} \\cdot T$")
    fig.savefig("out/intermediate_og_heatmaps.pdf", bbox_inches="tight")